In [30]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError("Add GEMINI_API_KEY in Colab Secrets.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [31]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
print("Secret found:", bool(key))
print("Key length:", len(key))

Secret found: True
Key length: 53


In [32]:
model="gemini-3.8-flash"

In [33]:
!pip -q install langgraph langchain-core google-genai
print("Packages installed")

Packages installed


In [34]:
import os
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini API key loaded from a secure secret")
else:
    client = None
    print("No key found. Demo mode is active. Add GEMINI_API_KEY for live API responses.")

Gemini API key loaded from a secure secret


In [35]:
SYSTEM_PROMPT = """You are SafeMap Support Assistant, a customer service assistant for SJSU SafeMap.
Help students submit safety reports, check fictional report statuses, and find appropriate safety resources.
Be calm, concise, and honest. Ask only for information needed to answer.
Do not invent a report status or claim that a report was submitted.
Pending reports require human analyst review before appearing on the public map.
For immediate danger, tell the user to contact emergency services or campus police directly.
This prototype does not replace emergency services.
"""

SAMPLE_REPORTS = {
    "SM-1042": "pending analyst review",
    "SM-2198": "approved and visible on the map",
    "SM-3011": "denied because the report was incomplete",
}

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    route: str
    route_history: list[str]

def latest_user_text(state):
    for message in reversed(state["messages"]):
        if isinstance(message, HumanMessage):
            return str(message.content)
    return ""

In [36]:
def demo_answer(agent, text):
    lower = text.lower()
    if agent == "status_agent":
        for report_id, status in SAMPLE_REPORTS.items():
            if report_id.lower() in lower:
                return f"Report {report_id} is currently {status}."
        return "Please provide a report number such as SM-1042 so I can check the sample status."
    if agent == "resource_agent":
        return "For a non-emergency concern, use SafeMap to submit a report and contact the appropriate SJSU campus safety office. If you are in immediate danger, contact emergency services or campus police directly."
    if "anonymous" in lower:
        return "You can mark the report anonymous. I still need the approximate location, date, time, and what happened. The report remains pending until an analyst reviews it."
    if "south entrance" in lower or "9:30" in lower:
        return "I noted the south entrance and approximately 9:30 PM. The next step is to confirm whether you want to submit the report anonymously."
    return "I can help you prepare a SafeMap report. Please share the approximate location, date and time, and a short description. Do not include unnecessary personal information."

def transcript(messages):
    lines = []
    for message in messages:
        speaker = "Student" if isinstance(message, HumanMessage) else "Assistant"
        lines.append(f"{speaker}: {message.content}")
    return "\n".join(lines)

def generate_answer(agent, state):
    text = latest_user_text(state)
    if client is None:
        return demo_answer(agent, text)
    role_prompts = {
        "report_agent": "You are the Report Support Agent. Help prepare a complete report and ask for missing details. Never claim submission.",
        "status_agent": "You are the Report Status Agent. Use only the sample report statuses and ask for a report ID when needed.",
        "resource_agent": "You are the Safety Resource Agent. Give safe, general guidance and direct immediate danger to emergency services or campus police.",
    }
    prompt = role_prompts[agent] + "\n\nConversation so far:\n" + transcript(state["messages"])
    response = client.models.generate_content(
        model=os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.2),
    )
    return response.text

In [37]:
def classify(state):
    text = latest_user_text(state).lower()
    if any(word in text for word in ["status", "pending", "approved", "denied", "report sm-"]):
        route = "status_agent"
    elif any(word in text for word in ["resource", "escort", "feel unsafe", "where can", "emergency"]):
        route = "resource_agent"
    else:
        route = "report_agent"
    return {"route": route, "route_history": [*state.get("route_history", []), route]}

def make_agent_node(agent):
    def node(state):
        return {"messages": [AIMessage(content=generate_answer(agent, state))]}
    return node

def choose_agent(state) -> Literal["report_agent", "status_agent", "resource_agent"]:
    return state["route"]

graph_builder = StateGraph(AgentState)
graph_builder.add_node("classify", classify)
graph_builder.add_node("report_agent", make_agent_node("report_agent"))
graph_builder.add_node("status_agent", make_agent_node("status_agent"))
graph_builder.add_node("resource_agent", make_agent_node("resource_agent"))
graph_builder.add_edge(START, "classify")
graph_builder.add_conditional_edges("classify", choose_agent)
graph_builder.add_edge("report_agent", END)
graph_builder.add_edge("status_agent", END)
graph_builder.add_edge("resource_agent", END)
graph = graph_builder.compile(checkpointer=MemorySaver())

def ask(text, thread_id):
    result = graph.invoke(
        {"messages": [HumanMessage(content=text)], "route_history": []},
        {"configurable": {"thread_id": thread_id}},
    )
    return result["route"], result["messages"][-1].content

In [38]:
import os

# Set the model environment variable to use the updated gemini-3.8-flash
os.environ["GEMINI_MODEL"] = "gemini-3.8-flash"

# Validation 1: three remembered turns in one conversation thread
thread = "safemap-report-demo"
conversation = [
    "I want to report suspicious activity near the SJSU library.",
    "It happened around 9:30 PM near the south entrance.",
    "Please keep my report anonymous.",
]
for i, text in enumerate(conversation, start=1):
    agent, answer = ask(text, thread)
    print(f"Turn {i} | {agent}\nStudent: {text}\nAssistant: {answer}\n")

# Validation 2: second distinct agent
agent, answer = ask("What is the status of report SM-1042?", "status-demo")
print(f"Status demo | {agent}\nAssistant: {answer}\n")

# Validation 3: third distinct agent
agent, answer = ask("What safety resource should I use if I feel unsafe walking at night?", "resource-demo")
print(f"Resource demo | {agent}\nAssistant: {answer}")

Turn 1 | report_agent
Student: I want to report suspicious activity near the SJSU library.
Assistant: If you are in immediate danger, please call 911 or SJSU University Police at (408) 924-2222 directly. This system does not replace emergency services.

To help draft your report for review by a safety analyst, please provide the following missing details:

1. **Specific Location:** Where near the library did this occur (e.g., exterior entrance, a specific side/street, or inside)?
2. **Date and Time:** When did you observe this activity?
3. **Description:** What specific behavior or activity did you observe, and are there any descriptions of individuals or vehicles involved?

Once you provide these details, we can assemble the draft for submission and review.

Turn 2 | report_agent
Student: It happened around 9:30 PM near the south entrance.
Assistant: Thank you. To complete your draft report for analyst review, please provide the remaining details:

1. **Date:** Did this occur today, o